# HouseholdBasketEnv — Baseline Eval (Phase 4.5 / Phase 5)

Run **vanilla Qwen2.5-3B-Instruct** (no fine-tuning) on the held-out seeds for Tasks 1/2/3. We record:

- per-seed total reward
- format-error rate (failed JSON / schema)
- terminal status (`+1.0` / `0.0` / `-0.5`)

Output is `docs/results/baseline.json` — the **must-beat** number for the trained agent (plan §10).

> **Hardware:** Local GPU or Colab T4. The first cell clones the repo and installs deps automatically.

## 1. Environment setup

The cell below clones `HouseholdBasketEnv` (if not already present) and installs all Python dependencies. Works on both local GPU machines and Colab.

In [ ]:
import os, sys, pathlib, subprocess

REPO_URL = "https://github.com/I-am-vishalmaurya/HouseholdBasketEnv"
CLONE_DIR = pathlib.Path("HouseholdBasketEnv")

# 1. Clone the repo if it doesn't exist alongside the notebook
if not CLONE_DIR.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)])
else:
    print(f"[bootstrap] {CLONE_DIR.resolve()} already exists; skipping clone.")

REPO_ROOT = CLONE_DIR.resolve()
ENV_ROOT  = REPO_ROOT / "household_basket_env"
RESULTS_DIR = REPO_ROOT / "docs" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ["HOUSEHOLD_BASKET_PRODUCTS_PATH"] = str(ENV_ROOT / "data" / "products.json")
os.environ.setdefault("HF_HOME", str(REPO_ROOT / ".hf_cache"))

print("REPO_ROOT   =", REPO_ROOT)
print("ENV_ROOT    =", ENV_ROOT)
print("RESULTS_DIR =", RESULTS_DIR)

In [ ]:
!pip -q install --upgrade pip
!pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install --no-deps "trl<0.20.0" peft accelerate bitsandbytes
!pip -q install pydantic==2.* fastapi uvicorn

import importlib, sys
for mod in ("household_basket_env", "household_basket_env.server.environment"):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from household_basket_env.server.environment import HouseholdBasketEnvironment
from household_basket_env.models import BasketAction, BasketObservation
from household_basket_env.server.curriculum import TIER_CONFIGS, load_verified_seeds

env = HouseholdBasketEnvironment()
print("Environment OK. Tiers:", list(TIER_CONFIGS.keys()))

## 2. Load the base model

We use **Qwen/Qwen2.5-3B-Instruct** loaded in 4-bit through Unsloth (matches the training notebook's model so the comparison is apples-to-apples). No LoRA / no fine-tune — this is the "vanilla prompting" baseline.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_SEQ_LEN = 2048
LOAD_IN_4BIT = True

# --- Try Unsloth fast-loader first; fall back to plain HF transformers ------
model, tokenizer = None, None
try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=LOAD_IN_4BIT,
        dtype=None,
    )
    FastLanguageModel.for_inference(model)
    print("Loaded via Unsloth.")
except Exception as e:
    print("Unsloth unavailable ->", e)
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    print("Loaded via transformers.")
print("Model ready.")

## 3. Prompt template + agent loop

The system prompt enforces the **`{"product_id": ..., "member_id": ..., "rationale": ...}`** schema (plan §3). Each step the agent sees:

- household summary (members, allergies, caps)
- catalog (product_id, category, meal_type, price, key nutrients)
- basket so far
- budget remaining + steps remaining

In [ ]:
SYSTEM_PROMPT = """You are a careful nutrition-aware shopping assistant for an Indian household. \
At each step you pick exactly ONE product from the catalog and assign it to ONE household member. \
You MUST output ONLY a JSON object with three keys: \"product_id\" (string), \"member_id\" (string), \"rationale\" (1 short sentence). \
Do NOT add any prose, code fences, or extra keys. Respect each member's caps and allergies; spread meal types across breakfast/lunch/dinner/snack/beverage; stay under the budget."""

import json, re
from textwrap import dedent

def render_observation(obs) -> str:
    members = []
    for m in obs.household:
        caps = ", ".join(f"{k}={v:.0f}" for k, v in m.thresholds_cap.items())
        members.append(f"- {m.member_id} | conds={m.conditions} | caps: {caps}")
    cands = []
    for c in obs.candidates[:50]:  # truncate to keep prompt small
        nuts = c.nutrition_per_100g
        cands.append(
            f"- {c.product_id} | {c.category} | meal={c.meal_type} | INR {c.price_inr:.0f} | "
            f"sugars_g={nuts.get('sugars_g',0):.1f} sodium_mg={nuts.get('sodium_mg',0):.0f} fat_g={nuts.get('fat_g',0):.1f}"
        )
    basket = []
    for t in obs.basket_so_far:
        basket.append(f"- {t.product_id} -> {t.member_id}")
    return dedent(f"""
    Household:
    {chr(10).join(members)}

    Catalog (top 50):
    {chr(10).join(cands)}

    Basket so far ({len(basket)}):
    {chr(10).join(basket) if basket else "(empty)"}

    Budget remaining: INR {obs.budget_remaining:.0f}
    Steps used: {obs.step_index} / {obs.max_steps}
    Attempts used: {obs.attempt_index} / {obs.max_steps * 4}

    Now pick ONE product and assign it to ONE member. Output JSON only.
    """).strip()

JSON_RE = re.compile(r"\{[^{}]*\}", re.S)

def extract_json(text: str):
    m = JSON_RE.search(text)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def chat_generate(system: str, user: str, max_new_tokens: int = 96, temperature: float = 0.2) -> str:
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    inputs = tokenizer.apply_chat_template(
        msgs, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    out = model.generate(
        inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=temperature > 0.0,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    txt = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
    return txt
print("Prompt + generation helpers ready.")

## 4. Run held-out eval per tier

We use the **`held_out_seeds`** lists generated by `seed_verifier`. For each seed we run one full episode, capping retries by the env's attempt-cap.

In [ ]:
from collections import Counter
from time import time

MAX_SEEDS_PER_TIER = 30  # subsample for fast-iteration; raise for final report
MAX_GENERATION_TOKENS = 96
TEMPERATURE = 0.2

def run_episode_with_model(env, seed: int, task_id: int):
    obs = env.reset(seed=seed, task_id=task_id)
    total_r = 0.0
    parse_fails = 0
    iter_cap = obs.max_steps * 4 + 2  # mirror attempt-cap
    last = obs
    for _ in range(iter_cap):
        if obs.done:
            break
        user_prompt = render_observation(obs)
        raw = chat_generate(SYSTEM_PROMPT, user_prompt, max_new_tokens=MAX_GENERATION_TOKENS, temperature=TEMPERATURE)
        payload = extract_json(raw)
        if payload is None:
            payload = raw  # let the env's parse-check catch it -> P_parse
            parse_fails += 1
        out = env.apply_raw_action(payload)
        total_r += out.reward or 0.0
        obs = out
        last = out
    return {
        "seed": seed,
        "task_id": task_id,
        "total_reward": round(total_r, 4),
        "parse_fails": parse_fails,
        "step_index": last.step_index,
        "attempt_index": last.attempt_index,
        "terminal_reward": last.terminal_reward,
        "basket_size": len(last.basket_so_far),
    }

records = {}
for task_id in (1, 2, 3):
    payload = load_verified_seeds(task_id)
    seeds = payload["held_out_seeds"][:MAX_SEEDS_PER_TIER]
    print(f"\n=== Task {task_id} | {len(seeds)} held-out seeds ===")
    rows = []
    t0 = time()
    for i, s in enumerate(seeds):
        r = run_episode_with_model(env, s, task_id)
        rows.append(r)
        if (i + 1) % 5 == 0:
            mean_r = sum(x["total_reward"] for x in rows) / len(rows)
            print(f"  [{i+1}/{len(seeds)}] mean_reward={mean_r:.3f} | elapsed={time()-t0:.0f}s")
    records[str(task_id)] = rows

print("\nDone. Aggregating ...")

In [ ]:
import statistics, json

summary = {}
for task_id, rows in records.items():
    rewards = [r["total_reward"] for r in rows]
    parse_rate = sum(1 for r in rows if r["parse_fails"] > 0) / max(1, len(rows))
    term_dist = Counter(round(r["terminal_reward"], 1) for r in rows)
    summary[task_id] = {
        "n_seeds": len(rows),
        "mean_reward": round(statistics.fmean(rewards), 4),
        "median_reward": round(statistics.median(rewards), 4),
        "stdev_reward": round(statistics.pstdev(rewards), 4) if len(rewards) > 1 else 0.0,
        "parse_failure_rate": round(parse_rate, 4),
        "terminal_reward_distribution": dict(term_dist),
    }

print("\nBASELINE SUMMARY")
print("================")
for k, v in summary.items():
    print(f"Task {k}: mean={v['mean_reward']:.3f} median={v['median_reward']:.3f} parse_fail={v['parse_failure_rate']:.2%} term={v['terminal_reward_distribution']}")

out_path = RESULTS_DIR / "baseline.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump({
        "model": MODEL_NAME,
        "load_in_4bit": LOAD_IN_4BIT,
        "max_seeds_per_tier": MAX_SEEDS_PER_TIER,
        "summary": summary,
        "rows": records,
    }, f, indent=2)
print(f"\nSaved -> {out_path}")

### Next steps

- Push `docs/results/baseline.json` to the repo so `eval_and_plots.ipynb` can compare.
- Move on to `training.ipynb` to start GRPO from this baseline.